# 16c — derive the count as `len(points)` instead of verbalising it

**The finding.** Alghisi et al. 2026 (*Getting to the Point*, arXiv:2603.21746), Qwen2.5-VL-7B
+ LoRA r=32/α=64, exact-match count accuracy on their OOD split (count-range extrapolation —
structurally our shape): direct count **23.41%**, point-then-count **verbalised** 14.04%,
**`#Coord` = `len(points)` 94.96%**. The lever is not pointing. It is **refusing to let the
model state the number** — taking the count as the length of its own coordinate list.

**Why rung 15 did not test this.** Rung 15 replicated Gautam's structured count *format* and
returned a null. And Gautam's Table I (re-verified from the repo PDF, p.5) shows the *joint*
count+point objective **costs** counting — counting-only 0.26 vs count+point 1.52. So the
structured target is the trick and the joint objective is the tax. `len(points)` is a third
thing: a decoding-side derivation.

⚠️ **The convention is a trap.** Qwen2-VL used `[0,1000]` → **Qwen2.5-VL switched to absolute
pixels** → **Qwen3-VL switched back to `[0,1000]`, points as `point_2d` in JSON**. Gautam and
Alghisi both ran Qwen2.5-VL, so *their exact serialisation is wrong for our backbone*. Arm
`a1` matches our native convention on purpose.

**Arms** (single variable = the answer target; image and question identical):
`a0` bare integer (control) · `a1` `point_2d` JSON, count = `len()` · `a2` numbered points
(Molmo), count = last index.

**Scored on** the *Clips* template — 681 val questions, 12 distinct true values, margin over
the template-aware floor only **+0.026**: the single largest hole in the exam. Read **margin**,
never raw accuracy.

🔴 **A null here is NOT decisive.** Alghisi report that prompt-only point-then-count is weak
without fine-tuning, and that a black-screen substitution costs <2% — the count is read off the
model's own text, not re-derived from pixels. This probe bounds the *prompt-only* path and
measures p99 at the longer `max_new_tokens`; it cannot bound the trained one.


In [ ]:
# --- bootstrap -----------------------------------------------------------------
import logging, os, sys, json
from pathlib import Path

REPO = Path("/workspace/repo")
sys.path.insert(0, str(REPO / "src"))
sys.path.insert(0, str(REPO / "vendor" / "orena-focus" / "src"))
sys.path.insert(0, str(Path.cwd() / "_models"))          # experiment-private glue

os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
# --- parameters (RAW LITERALS ONLY; papermill injects overrides directly below) --
SMOKE = True
SEED  = 0

RUN        = "16c_len_points_v1"
DATA_ROOT  = "/workspace/orena-data"
# rung 06 ep3 — the current best checkpoint and the epoch-matched control (bucket_mean 0.5724)
MODEL_PATH = "/workspace/repo/experiments/06-vit-lora/runs/06_vit_lora_v1/merged/checkpoint-2580"

TEMPLATE      = r"how many\s+clips"   # the 681-question hole; widen only with a reason
ARMS          = ("a0", "a1", "a2")
N_ITEMS_SMOKE = 24


In [ ]:
# --- derived (MUST live below the parameters cell) -------------------------------
DATA_ROOT  = Path(DATA_ROOT)
MODEL_PATH = Path(MODEL_PATH)
RUN_DIR    = Path.cwd() / "runs" / RUN
N_ITEMS    = N_ITEMS_SMOKE if SMOKE else None   # None = the whole template
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("run dir:", RUN_DIR, "| SMOKE:", SMOKE, "| arms:", ARMS, "| n_items:", N_ITEMS or "ALL")


In [ ]:
# --- select the scored items (gold only; no model loaded yet) --------------------
from frame.config import BaselineConfig
from frame.data import load_frame_items
import len_points as lp

cfg   = BaselineConfig(data_root=DATA_ROOT)
items = load_frame_items(cfg, splits=("test",))
sel   = lp.select_number_items(items, template_re=TEMPLATE)
if N_ITEMS:
    sel = sel[:N_ITEMS]

import collections
golds = [g for _, g in sel]
assert sel, f"no items matched {TEMPLATE!r} — check the template regex against the corpus"
print("items:", len(sel), "| distinct golds:", sorted(set(golds)))
print("distribution:", collections.Counter("OOD" if it.dataset == "heico" else "ID" for it, _ in sel))
print("trivial floor (always answer the mode):", round(lp.template_floor(golds), 4))


In [ ]:
# --- run the three arms on ONE model load ---------------------------------------
# All three arms differ only in the prompt suffix and max_new_tokens, so a single load
# serves them all — and keeps the image path byte-identical across arms.
import time, gc
from frame.engine import QwenFrameEngine

rows = []
for arm in ARMS:
    mcfg = BaselineConfig(data_root=DATA_ROOT, model_path=MODEL_PATH,
                          max_new_tokens=lp.ARM_MAX_NEW_TOKENS[arm])
    eng = QwenFrameEngine(mcfg); eng.load()
    t0 = time.time()
    rows += lp.run_arm(eng, sel, arm)
    dt = time.time() - t0
    print(f"arm {arm}: {len(sel)} questions in {dt:.0f}s  ({dt/max(len(sel),1):.3f} s/q)")
    del eng; gc.collect(); torch.cuda.empty_cache()


In [ ]:
# --- score: MARGIN over the template floor, never raw accuracy -------------------
import pandas as pd, json

df = pd.DataFrame(rows); df.to_csv(RUN_DIR / "rows.csv", index=False)
res = lp.score(rows); (RUN_DIR / "score.json").write_text(json.dumps(res, indent=2))
print(pd.DataFrame(res).T.round(4).to_string())
print()
print("parse methods:", df.groupby(["arm", "method"]).size().to_dict())
print("Alghisi reference (Qwen2.5-VL-7B, their OOD): direct 0.2341 | verbalised 0.1404 | len() 0.9496")


In [ ]:
# --- eyeball: what did each arm actually emit? (user rule: error examples every run) ---
for arm in ARMS:
    sub = df[df.arm == arm].head(4)
    print(f"--- {arm} ---")
    for r in sub.itertuples():
        print(f"  gold={r.gold} pred={r.value} ok={r.correct} | {str(r.raw)[:110]!r}")
